# 01 — Hello, runner

Execute a notebook through `agent_kernel.runtime.notebook_runner.NotebookRunner` and inspect the JSONL provenance ledger it produced.

This is the smallest useful thing the system does: take a notebook on disk, execute it via `nbclient` against a real Jupyter kernel, capture per-cell provenance, and write an executed-notebook artifact to a run-scoped directory.

**Kernel:** standard `python3`. We use `agent_kernel` purely as a library here.

In [ ]:
import json, tempfile, shutil
from pathlib import Path

from agent_kernel.runtime.notebook_runner import NotebookRunner
from agent_kernel.storage import JSONLEventStore, WorkspaceLayout
from agent_kernel.util import new_id

# Use a throwaway workspace so this notebook is hermetic and re-runnable.
workspace = Path(tempfile.mkdtemp(prefix='ak-ex01-'))
ws = WorkspaceLayout(workspace); ws.ensure()
print('workspace:', workspace)

## 1. Author a tiny notebook on disk

Anything `nbclient` can execute will work. We use three cells: a normal computation, a print, and a final expression so we can see outputs.

In [ ]:
import nbformat
from nbformat.v4 import new_notebook, new_code_cell

target = workspace / 'hello.ipynb'
nbformat.write(new_notebook(
    cells=[
        new_code_cell('x = 6 * 7'),
        new_code_cell("print(f'the answer is {x}')"),
        new_code_cell('x'),
    ],
    metadata={'kernelspec': {'name': 'python3', 'display_name': 'Python 3'}},
), target)
print('wrote', target)

## 2. Run it through `NotebookRunner`

The runner takes an event store and a runs directory; for every nbclient hook it emits a typed `ProvenanceEvent` to the JSONL ledger and writes the executed notebook + captured stdout/stderr to `runs/<run_id>/`.

In [ ]:
events = JSONLEventStore(ws.events_dir, fsync=True)
runner = NotebookRunner(events, ws.runs_dir)
result = runner.run(target, task_id=new_id('task'), kernel_name='python3', timeout=30)
print('run_id:        ', result.run_id)
print('task_id:       ', result.task_id)
print('executed file: ', result.executed_notebook_path)
print('exists:        ', Path(result.executed_notebook_path).exists())

## 3. Inspect the executed notebook

The third cell evaluated `x`, so its output is `42`. The runner preserves outputs exactly as `nbclient` produced them.

In [ ]:
executed = nbformat.read(result.executed_notebook_path, as_version=4)
for i, cell in enumerate(executed.cells):
    outs = [o.get('text') or o.get('data', {}).get('text/plain') for o in cell.get('outputs', [])]
    print(f'cell {i}: source={cell.source!r:40s}  outputs={outs}')

## 4. Walk the JSONL ledger

Every event the runner emitted is on disk as a single JSON line in `<workspace>/.agent_kernel/events/YYYY-MM-DD.jsonl`. The event types you'll see for a clean run are: `task.created` (synthesized for the runner's task scope), `notebook.execution.started`, one `cell.execution.started` + `cell.execution.completed` per code cell, `notebook.execution.completed`, and `task.completed`.

In [ ]:
lines = []
for f in sorted(ws.events_dir.glob('*.jsonl')):
    lines.extend(f.read_text().splitlines())
print(f'{len(lines)} events on disk')
for ln in lines:
    e = json.loads(ln)
    print(f"{e['ts']}  {e['event_type']:35s}  {e.get('task_id','-')}")

## 5. Same thing from the CLI

If you don't need to embed this in Python code, the same flow is one shell command:

```bash
agent-kernel run hello.ipynb --workspace .
```

This is exercised in `tests/integration/test_m2_notebook_runner.py`.

In [ ]:
# Tidy up the temp workspace.
shutil.rmtree(workspace, ignore_errors=True)
print('cleaned')